# 01 — Exploratory Data Analysis (EDA)
### Smart Agriculture Decision Support System

**Phase 4** — theo `Phase_DSS_Agriculture.md`.

**Input:** `data/processed/metadata_clean.csv` (54,284 dòng, sinh ra từ Phase 3, đã loại 21 ảnh trùng lặp).

**Checklist Phase 4:**
- [ ] Phân bố class
- [ ] Kích thước ảnh
- [ ] Duplicate Analysis (xác nhận lại kết quả Phase 3)
- [ ] Missing Analysis
- [ ] Visualization

> **Lưu ý:** Notebook này import các hàm tính toán từ `src/data/eda_analyzer.py` — logic tính toán nằm trong `src/` theo Modular Design của SDS, notebook chỉ gọi hàm và trực quan hóa (Matplotlib/Plotly).

In [ ]:
import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    """Dò project root bằng cách tìm thư mục cha gần nhất chứa configs/."""
    for parent in [start] + list(start.parents):
        if (parent / "configs").is_dir():
            return parent
    raise RuntimeError("Không tìm thấy project root (thư mục chứa configs/).")


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
from PIL import Image

from src.config.config_loader import get_project_root, load_config
from src.data.eda_analyzer import (
    load_clean_metadata,
    compute_class_distribution,
    compute_plant_type_distribution,
    compute_healthy_vs_diseased,
    compute_imbalance_ratio,
    compute_image_dimension_stats,
    compute_duplicate_summary,
    compute_missing_analysis,
    build_eda_summary,
    generate_eda_report,
)

pd.set_option("display.max_rows", 50)

## 1. Tổng quan dataset

In [ ]:
df = load_clean_metadata()
print(f"Số dòng: {len(df):,} | Số cột: {df.shape[1]}")
df.head()

In [ ]:
df.dtypes

## 2. Phân bố Class

Phase 3 đã phát hiện mất cân bằng ~36x giữa `Orange___Haunglongbing_(Citrus_greening)` (5,507 ảnh) và `Potato___healthy` (152 ảnh). Phần này trực quan hóa chi tiết theo class, theo plant_type, và theo healthy/diseased.

In [ ]:
class_distribution = compute_class_distribution(df)
class_distribution

In [ ]:
fig = px.bar(
    class_distribution,
    x="image_count",
    y="class_name",
    orientation="h",
    title="Phân bố số lượng ảnh theo Class",
    height=900,
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

In [ ]:
plant_type_distribution = compute_plant_type_distribution(df)
fig = px.bar(
    plant_type_distribution,
    x="plant_type",
    y="image_count",
    title="Phân bố số lượng ảnh theo Plant Type",
)
fig.update_xaxes(tickangle=45)
fig.show()

In [ ]:
healthy_vs_diseased = compute_healthy_vs_diseased(df)
fig = px.pie(
    healthy_vs_diseased,
    names="status",
    values="image_count",
    title="Tỷ lệ ảnh Healthy vs Diseased",
)
fig.show()

In [ ]:
imbalance_ratio = compute_imbalance_ratio(class_distribution)
most_common = class_distribution.iloc[0]
least_common = class_distribution.iloc[-1]

print(f"Tỷ lệ mất cân bằng (max/min): {imbalance_ratio}x")
print(f"Class nhiều ảnh nhất: {most_common['class_name']} ({most_common['image_count']:,} ảnh)")
print(f"Class ít ảnh nhất: {least_common['class_name']} ({least_common['image_count']:,} ảnh)")

## 3. Kích thước ảnh (Image Dimensions)

`width`/`height` đã có sẵn trong `metadata_clean.csv` từ Phase 3 — không cần đọc lại ảnh gốc.

In [ ]:
dimension_stats = compute_image_dimension_stats(df)
dimension_stats

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df["width"], bins=50)
axes[0].set_title("Phân bố Width")
axes[1].hist(df["height"], bins=50)
axes[1].set_title("Phân bố Height")
plt.tight_layout()
plt.show()

In [ ]:
SCATTER_SAMPLE_SIZE = 5000
sample_df = df.sample(n=min(SCATTER_SAMPLE_SIZE, len(df)), random_state=42)

fig = px.scatter(
    sample_df,
    x="width",
    y="height",
    color="plant_type",
    opacity=0.4,
    title=f"Width vs Height theo Plant Type (mẫu ngẫu nhiên {SCATTER_SAMPLE_SIZE:,} ảnh)",
)
fig.show()

## 4. Duplicate Analysis (xác nhận lại kết quả Phase 3)

Việc tính hash MD5 và loại 21 ảnh trùng lặp đã hoàn thành ở Phase 3 (~18 phút chạy). Ở đây chỉ kiểm tra chéo (sanity check) — không tính lại hash.

In [ ]:
duplicate_summary = compute_duplicate_summary(df)
print(duplicate_summary)

assert duplicate_summary["remaining_duplicates"] == 0, (
    "Phát hiện duplicate còn sót lại trong metadata_clean.csv — kiểm tra lại Phase 3!"
)
print("✅ Xác nhận: không còn ảnh trùng lặp trong metadata_clean.csv")

## 5. Missing Analysis

In [ ]:
missing_analysis = compute_missing_analysis(df)
if missing_analysis.empty:
    print("✅ Không phát hiện giá trị thiếu (NaN) ở bất kỳ cột nào.")
else:
    display(missing_analysis)

## 6. (Tuỳ chọn) Ảnh mẫu theo Plant Type

> **Giả định:** cột chứa đường dẫn ảnh tương đối tên là `image_path`. Nếu `metadata_schema.md` dùng tên cột khác, sửa lại `IMAGE_PATH_COLUMN` bên dưới. Ô này có thể bỏ qua an toàn nếu không tìm thấy cột — không ảnh hưởng các phần còn lại.

In [ ]:
IMAGE_PATH_COLUMN = "image_path"
RAW_DATA_DIR = get_project_root() / load_config("base")["paths"]["data_raw_dir"]

if IMAGE_PATH_COLUMN in df.columns:
    sample_plant_types = df["plant_type"].unique()[:6]
    fig, axes = plt.subplots(1, len(sample_plant_types), figsize=(18, 3))
    for ax, plant_type in zip(axes, sample_plant_types):
        row = df[df["plant_type"] == plant_type].sample(1, random_state=42).iloc[0]
        image_path = RAW_DATA_DIR / row[IMAGE_PATH_COLUMN]
        try:
            image = Image.open(image_path)
            ax.imshow(image)
            ax.set_title(plant_type, fontsize=9)
        except FileNotFoundError:
            ax.set_title(f"{plant_type}\n(không đọc được ảnh)", fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print(
        f"Bỏ qua: không tìm thấy cột '{IMAGE_PATH_COLUMN}' trong metadata. "
        "Kiểm tra lại tên cột trong docs/data_dictionary/metadata_schema.md."
    )

## 7. Tổng hợp & Xuất báo cáo

Sinh `eda_report.md` + các file thống kê (`class-distribution.csv`, `plant-type-distribution.csv`, `image-dimension-stats.csv`, `missing-analysis.csv`) vào `docs/data_dictionary/`.

In [ ]:
base_config = load_config("base")
data_config = load_config("data")
eda_config = data_config["eda"]
docs_dir = get_project_root() / base_config["paths"]["docs_data_dictionary_dir"]

summary = build_eda_summary(
    df, imbalance_warning_threshold=eda_config["imbalance_warning_threshold"]
)

generate_eda_report(summary, docs_dir / eda_config["report_filename"])
class_distribution.to_csv(docs_dir / eda_config["class_distribution_filename"], index=False)
plant_type_distribution.to_csv(
    docs_dir / eda_config["plant_type_distribution_filename"], index=False
)
dimension_stats.to_csv(docs_dir / eda_config["image_dimension_stats_filename"])
missing_analysis.to_csv(docs_dir / eda_config["missing_analysis_filename"])

print("✅ Đã sinh eda_report.md và các file thống kê vào docs/data_dictionary/")

## Kết luận

- Điền lại số liệu thực tế sau khi chạy notebook này (tổng số ảnh, số class, tỷ lệ mất cân bằng, kết quả missing analysis) — xem `docs/data_dictionary/eda_report.md` vừa sinh ra.
- **Khuyến nghị cho Phase 5/6:** xử lý mất cân bằng class bằng class weighting, oversampling, hoặc augmentation có trọng số; dùng stratified split khi chia train/val/test để các class ít ảnh vẫn có đại diện đầy đủ ở cả 3 tập.
- **Bước tiếp theo:** Phase 5 — Data Preprocessing (resize, normalize, dataset split, augmentation, DataLoader) theo `Phase_DSS_Agriculture.md`.